# Project 04 — Freddie Mac IFRS 9 Stress Test (Real Data)

## Objective
Apply the IFRS 9 framework to a real-world mortgage portfolio spanning 2006 to 2026. 
This dataset includes the 2008 Global Financial Crisis (GFC) and the 2020 COVID-19 pandemic, allowing us to analyze:
1. **Vintage Risk:** The subprime crisis.
2. **Policy Interventions (Forbearance):** How COVID moratoriums artificially suppressed defaults.
3. **Expected Credit Loss (ECL):** Calculating provisions using dynamic LTV (ELTV).


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE = Path(r"C:\CreditRiskCourse")
DATA = BASE / "data/processed/freddie_stratified_sample.parquet"
FIG  = BASE / "outputs/figures"
SUM  = BASE / "outputs/summaries"
FIG.mkdir(parents=True, exist_ok=True)
SUM.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("Environment ready.")

## 1. Data Hygiene & Cleaning

**The "Deloitte" Audit:** Before modeling, we must clean the data.
- We map the string `CURRENT LOAN DELINQUENCY STATUS` into numeric DPD buckets.
- We handle missing values (e.g., `ELTV`, `DTI`).
- We extract the origination year from `FIRST PAYMENT DATE` to establish vintages.


In [ ]:
# Load a 10% sample of the parquet for speed in this notebook (you can remove this for the final run)
df = pd.read_parquet(DATA)

print(f"Loaded {len(df):,} rows.")

# Clean Delinquency Status (Map to numeric DPD)
# Freddie Mac codes: '00'=Current, '01'=30-59, '02'=60-89, '03'=90-119, etc. 'RA'=REO/Foreclosure
def map_dpd(status):
    if pd.isna(status): return np.nan
    if status in ['00', '0']: return 0
    if status == 'RA': return 120 # REO Acquisition (Default)
    try:
        val = int(float(status))
        if val >= 3: return 90 # 90+ DPD
        if val == 2: return 60
        if val == 1: return 30
        return 0
    except:
        return 0

df["dpd_days"] = df["CURRENT LOAN DELINQUENCY STATUS"].apply(map_dpd)

# Extract Origination Year (Vintage)
df["origination_year"] = (df["FIRST PAYMENT DATE"] // 100).astype(int)

# Handle Missing Values (Simple Imputation for demonstration)
df["CLASSIC FICO"] = df["CLASSIC FICO"].fillna(df["CLASSIC FICO"].median())
df["ORIGINAL DEBT-TO-INCOME (DTI) RATIO"] = df["ORIGINAL DEBT-TO-INCOME (DTI) RATIO"].fillna(df["ORIGINAL DEBT-TO-INCOME (DTI) RATIO"].median())
df["ESTIMATED LOAN-TO-VALUE (ELTV)"] = df["ESTIMATED LOAN-TO-VALUE (ELTV)"].fillna(df["ORIGINAL LOAN-TO-VALUE (LTV)"])

print("Data cleaned. DPD buckets and Origination Years created.")
print(df[["PERIOD", "origination_year", "dpd_days", "default_flag", "CLASSIC FICO"]].head())

## 2. Time-Based EDA: Vintage Analysis

**The Maturity Bias Trap:** If we look at total defaults, newer loans look "safer" simply because they haven't had enough time to default yet. 
By tracking the default rate by **Origination Year**, we visually prove how catastrophic the 2006/2007 subprime vintages were compared to the stricter underwriting of 2015+.


In [ ]:
# Calculate default rate by Vintage Year
vintage_risk = df.groupby("origination_year")["default_flag"].mean().reset_index()
vintage_risk["default_rate_%"] = vintage_risk["default_flag"] * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=vintage_risk, x="origination_year", y="default_rate_%", palette="Reds_r", ax=ax)
ax.set_title("Default Rate by Origination Vintage (2006-2026)")
ax.set_ylabel("Lifetime Default Rate (%)")
ax.set_xlabel("Origination Year")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG / "04_vintage_analysis.png", dpi=120)
plt.show()
print("Notice the massive spike in the 2006-2008 vintages (GFC).")

## 3. Policy Intervention: The COVID-19 Forbearance

**The Regulatory Blindspot:** In 2020, the US government passed the CARES Act. Lenders were required to offer forbearance (payment pauses). 
Look at the chart below. The economy crashed in 2020, but defaults (RA/Foreclosure) dropped to near zero. **If a model doesn't account for policy interventions, it will underestimate risk.**


In [ ]:
# Calculate monthly default rate across the entire portfolio
monthly_risk = df.groupby("PERIOD")["default_flag"].mean().reset_index()
monthly_risk["default_rate_%"] = monthly_risk["default_flag"] * 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_risk["PERIOD"], monthly_risk["default_rate_%"], color="darkred", linewidth=2)

# Annotations
ax.axvspan(pd.Timestamp("2007-01-01"), pd.Timestamp("2009-12-31"), color="red", alpha=0.1, label="2008 GFC")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-12-31"), color="blue", alpha=0.1, label="COVID Forbearance")

ax.set_title("Monthly Default Rate (2006-2026) - Policy Interventions Annotated")
ax.set_ylabel("Default Rate (%)")
ax.set_xlabel("Period")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(FIG / "04_policy_intervention.png", dpi=120)
plt.show()

## 4. LGD Modeling using Dynamic LTV

Loss Given Default is heavily dependent on house prices. When the ELTV (Estimated LTV) exceeds 100%, the borrower is underwater, and the bank loses money on foreclosure. We map LGD dynamically based on the ELTV.


In [ ]:
def calc_lgd(row):
    eltv = row["ESTIMATED LOAN-TO-VALUE (ELTV)"]
    if pd.isna(eltv): eltv = row["ORIGINAL LOAN-TO-VALUE (LTV)"]
    
    if eltv > 100: return 0.50  # Underwater, but bank recovers some via foreclosure
    elif eltv > 80: return 0.30 # High risk
    elif eltv > 60: return 0.15 # Moderate
    else: return 0.05           # Highly equity-rich, very safe

df["lgd"] = df.apply(calc_lgd, axis=1)
df["ead"] = df["CURRENT ACTUAL UPB"]

print("Portfolio Weighted Average LGD: {:.2%}".format((df["lgd"] * df["ead"]).sum() / df["ead"].sum()))

## 5. IFRS 9 Stress Test

We simulate an IFRS 9 framework by assuming a PD shock. For the GFC (2008) and COVID (2020), we apply a **Stress Multiplier** to the base PD (derived from the FICO score) to see how ECL behaves.


In [ ]:
# Base PD proxy (we use a simple FICO-based heuristic for demonstration)
def fico_to_pd(fico):
    if fico >= 750: return 0.01
    elif fico >= 700: return 0.03
    elif fico >= 650: return 0.08
    elif fico >= 600: return 0.15
    else: return 0.25

df["base_pd"] = df["CLASSIC FICO"].apply(fico_to_pd)

# Define Scenarios
scenarios = {
    "Base (Normal)": 1.0,
    "2008 GFC (Severe)": 3.0,   # 3x default risk
    "2020 COVID (Forbearance)": 1.5 # 1.5x risk, but muted by policy
}

results = []
for name, mult in scenarios.items():
    df["stressed_pd"] = (df["base_pd"] * mult).clip(0, 1)
    df["ecl"] = df["stressed_pd"] * df["lgd"] * df["ead"]
    results.append({"Scenario": name, "Total ECL": df["ecl"].sum()})

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=res_df, x="Scenario", y="Total ECL", palette="viridis", ax=ax)
ax.set_title("IFRS 9 Expected Credit Loss (ECL) under Scenarios")
ax.set_ylabel("ECL ($)")
plt.tight_layout()
plt.savefig(FIG / "04_ecl_scenarios.png", dpi=120)
plt.show()